# Paper ZALL degradation histograms (cleaner + regressor)

This notebook reproduces the usual cleaner/regressor metrics, but shows them as grouped histograms versus:
- SNR
- NBS tag parsed from IRF name (`N`, `B`, `S`)
- zenith (`z20`, `z40`, `z60`)
- theta/alpha/source offset from FoV center

Only the **ZALL** models are used.


In [1]:
import pickle
import re
from os.path import join, expandvars

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import astropy.units as u

from astropy.coordinates import SkyCoord
from astroai.tools.utils import (
    split_noisy_dataset,
    split_regression_dataset,
    create_circular_mask,
    set_wcs,
)


2026-04-24 13:35:33.791314: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-24 13:35:39.345324: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2026-04-24 13:35:39.345349: I tensorflow/compiler/xla/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
2026-04-24 13:35:56.328350: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory
2026-

In [5]:
# plotting defaults
FIGSIZE = (9, 6)
FS = 14
N_BINS_HIST = 30

# cleaner ON-region setup (same style used in paper notebooks)
radius_deg = 0.2 * u.deg
pixelsize = 0.025 * u.deg
point_ref = 100
radius_pix = (radius_deg / pixelsize).to_value('')

# data/model configuration (ZALL only)
ROOT = f"{expandvars('$HOME')}/astroAI/astroai"
DATA_ROOT = f"{expandvars('$HOME')}/E4/irf_random/crab"

CLEANER_TABLE = 'cleaner_5sgm_expALL.pickle'
CLEANER_MODEL = 'cleaner_zALL.keras'

REGRESSOR_TABLE = 'regressor_5sgm_xy_flip.pickle'
REGRESSOR_MODEL = 'regressor_zALL.keras'
BINNING = 200


In [6]:
def load_dataset(path):
    if path.endswith('.pickle'):
        with open(path, 'rb') as f:
            return pickle.load(f)
    if path.endswith('.npy'):
        return np.load(path, allow_pickle=True, encoding='latin1', fix_imports=True).flat[0]
    raise ValueError(f'Unsupported dataset extension: {path}')


def guess_col(df, candidates, required=True):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f'None of columns found: {candidates}')
    return None


def parse_zenith(irf_name):
    txt = str(irf_name).lower()
    m = re.search(r'z(20|40|60)', txt)
    return f"z{m.group(1)}" if m else 'z?'


def parse_nbs(irf_name):
    txt = str(irf_name)
    # Convention in IRF names often uses _N_ and _S_.
    # If neither is present, we tag as B (baseline/no explicit N/S tag).
    if '_N_' in txt:
        return 'N'
    if '_S_' in txt:
        return 'S'
    return 'B'


def add_common_meta(df):
    out = df.copy()

    irf_col = guess_col(out, ['irf', 'IRF'], required=False)
    if irf_col is None:
        out['irf'] = 'unknown'
        irf_col = 'irf'

    out['zenith_tag'] = out[irf_col].apply(parse_zenith)
    out['nbs_tag'] = out[irf_col].apply(parse_nbs)

    theta_col = guess_col(out, ['theta', 'Theta', 'offset', 'source_offset', 'src_offset', 'alpha'], required=False)
    out['theta_tag'] = out[theta_col] if theta_col else np.nan

    snr_col = guess_col(out, ['snr', 'SNR'], required=False)
    if snr_col is None:
        ex_col = guess_col(out, ['excess', 'counts_excess'], required=False)
        off_col = guess_col(out, ['counts_off', 'off', 'background'], required=False)
        if ex_col and off_col:
            ex = out[ex_col].to_numpy(dtype=float)
            off = out[off_col].to_numpy(dtype=float)
            out['snr_tag'] = ex / np.sqrt(np.maximum(ex + off, 1e-12))
        else:
            out['snr_tag'] = np.nan
    else:
        out['snr_tag'] = out[snr_col]

    return out


def make_quantile_bins(series, n=4, label='q'):
    s = pd.Series(series).astype(float)
    s = s.replace([np.inf, -np.inf], np.nan).dropna()
    if s.empty:
        return pd.Series([], dtype=str), []
    n_unique = s.nunique()
    n_eff = max(1, min(n, n_unique))
    qcats = pd.qcut(s, q=n_eff, duplicates='drop')
    labels = [f"{label}{i+1}: {interval.left:.3g}-{interval.right:.3g}" for i, interval in enumerate(qcats.cat.categories)]
    mapper = dict(zip(qcats.cat.categories, labels))
    return qcats.map(mapper), labels


def grouped_hist(ax, values, groups, title, xlabel, bins=N_BINS_HIST):
    v = pd.Series(values).astype(float)
    g = pd.Series(groups)
    mask = np.isfinite(v) & g.notna()
    v = v[mask]
    g = g[mask]

    labels = list(pd.unique(g))
    data = [v[g == lab].to_numpy() for lab in labels]
    labels = [f"{lab} (n={len(d)})" for lab, d in zip(labels, data)]

    ax.hist(data, bins=bins, histtype='step', density=False, label=labels)
    ax.set_title(title, fontsize=FS)
    ax.set_xlabel(xlabel, fontsize=FS)
    ax.set_ylabel('samples in dataset', fontsize=FS)
    ax.grid(alpha=0.3)
    ax.tick_params(axis='both', labelsize=FS-2)
    ax.legend(fontsize=FS-4)


In [ ]:
# --- cleaner (ZALL) ---
cleaner_path = join(DATA_ROOT, CLEANER_TABLE)
cleaner_ds = load_dataset(cleaner_path)
cleaner_info = pd.read_csv(join(DATA_ROOT, CLEANER_TABLE.replace('.pickle', '.dat')), sep=' ', header=0).sort_values(by=['seed'])

train_noisy, train_clean, test_noisy, test_clean = split_noisy_dataset(
    cleaner_ds, split=80, reshape=True, binning=200
)

cleaner_model = tf.keras.models.load_model(join(ROOT, 'models/crta_models', CLEANER_MODEL))
cleaner_pred = cleaner_model.predict(test_noisy)

In [13]:
# align info rows with test split using seed convention from existing notebooks
seed_start = len(train_noisy) + 1
seed_stop = seed_start + len(test_noisy)
cleaner_meta = cleaner_info[cleaner_info['seed'].between(seed_start, seed_stop - 1)].copy()
cleaner_meta = cleaner_meta.sort_values('seed').reset_index(drop=True)

# fallback if seed filtering returns unexpected length
if len(cleaner_meta) != len(test_noisy):
    cleaner_meta = cleaner_info.iloc[len(train_noisy):len(train_noisy) + len(test_noisy)].copy().reset_index(drop=True)

cleaner_meta = add_common_meta(cleaner_meta)

# same metrics used in paper cleaner notebook
sum_residual_std = []
sum_residual_cnn = []
sum_on_std = []
sum_on_cnn = []

print(pixelsize)
break

for i, (noisy, clean, pred) in enumerate(zip(test_noisy, test_clean, cleaner_pred)):
    res_std = noisy - clean
    res_cnn = noisy - pred

    sum_residual_std.append(np.sum(res_std))
    sum_residual_cnn.append(np.sum(res_cnn))

    row = cleaner_meta.iloc[i]
    w = set_wcs(
        point_ra=row['point_ra'],
        point_dec=row['point_dec'],
        point_ref=point_ref,
        pixelsize=pixelsize,
    )
    x, y = w.world_to_pixel(
        SkyCoord(row['source_ra'], row['source_dec'], unit='deg', frame='icrs')
    )

    h, wdim = clean.shape[:2]
    mask = create_circular_mask(h, wdim, center=(y, x), radius=radius_pix)

    masked_std = clean.copy()
    masked_std[~mask] = 0

    masked_cnn = pred.copy()
    masked_cnn[~mask] = 0

    sum_on_std.append(np.sum(masked_std))
    sum_on_cnn.append(np.sum(masked_cnn))

cleaner_metrics = cleaner_meta.copy()
cleaner_metrics['sum_residual_std'] = sum_residual_std
cleaner_metrics['sum_residual_cnn'] = sum_residual_cnn
cleaner_metrics['sum_residual_diff'] = cleaner_metrics['sum_residual_std'] - cleaner_metrics['sum_residual_cnn']
cleaner_metrics['sum_on_std'] = sum_on_std
cleaner_metrics['sum_on_cnn'] = sum_on_cnn
cleaner_metrics['sum_on_diff'] = cleaner_metrics['sum_on_std'] - cleaner_metrics['sum_on_cnn']

cleaner_metrics.head()

0.025 deg


SyntaxError: 'break' outside loop (2019080436.py, line 20)

In [ ]:
# --- regressor (ZALL) ---
reg_path = join(DATA_ROOT, REGRESSOR_TABLE)
reg_ds = load_dataset(reg_path)
reg_info = pd.read_csv(join(DATA_ROOT, REGRESSOR_TABLE.replace('.pickle', '.dat')), sep=' ', header=0).sort_values(by=['seed'])

train_data, train_labels, test_data, test_labels = split_regression_dataset(
    reg_ds, split=80, reshape=True, binning=BINNING
)

reg_model = tf.keras.models.load_model(join(ROOT, 'models/crta_models', REGRESSOR_MODEL))
reg_pred = reg_model.predict(test_data) * BINNING

seed_start = len(train_data) + 1
seed_stop = seed_start + len(test_data)
reg_meta = reg_info[reg_info['seed'].between(seed_start, seed_stop - 1)].copy()
reg_meta = reg_meta.sort_values('seed').reset_index(drop=True)

if len(reg_meta) != len(test_data):
    reg_meta = reg_info.iloc[len(train_data):len(train_data) + len(test_data)].copy().reset_index(drop=True)

reg_meta = add_common_meta(reg_meta)

reg_errors_deg = []
for i, (pred, label) in enumerate(zip(reg_pred, test_labels)):
    row = reg_meta.iloc[i]
    w = set_wcs(
        point_ra=row['point_ra'],
        point_dec=row['point_dec'],
        point_ref=point_ref,
        pixelsize=pixelsize,
    )
    found_sky = w.pixel_to_world(pred[0], pred[1])
    true_sky = SkyCoord(row['source_ra'], row['source_dec'], unit='deg', frame='icrs')
    reg_errors_deg.append(true_sky.separation(found_sky).degree)

reg_metrics = reg_meta.copy()
reg_metrics['err_deg'] = reg_errors_deg
reg_metrics.head()


In [ ]:
# build grouping columns used across cleaner and regressor
for df in [cleaner_metrics, reg_metrics]:
    df['group_nbs'] = df['nbs_tag']
    df['group_zenith'] = pd.Categorical(df['zenith_tag'], categories=['z20', 'z40', 'z60', 'z?'], ordered=True)

    snr_q, _ = make_quantile_bins(df['snr_tag'], n=4, label='SNR')
    theta_q, _ = make_quantile_bins(df['theta_tag'], n=4, label='theta')
    df['group_snr'] = snr_q
    df['group_theta'] = theta_q

print('Cleaner groups:', cleaner_metrics[['group_snr', 'group_nbs', 'group_zenith', 'group_theta']].notna().mean())
print('Regressor groups:', reg_metrics[['group_snr', 'group_nbs', 'group_zenith', 'group_theta']].notna().mean())


In [ ]:
# CLEANER: residual degradation views
fig, axs = plt.subplots(2, 2, figsize=(16, 12))

grouped_hist(axs[0, 0], cleaner_metrics['sum_residual_diff'], cleaner_metrics['group_snr'],
             'Cleaner residual diff vs SNR bins', 'sum residual (STD - CNN) counts')
grouped_hist(axs[0, 1], cleaner_metrics['sum_residual_diff'], cleaner_metrics['group_nbs'],
             'Cleaner residual diff vs NBS tag', 'sum residual (STD - CNN) counts')
grouped_hist(axs[1, 0], cleaner_metrics['sum_residual_diff'], cleaner_metrics['group_zenith'],
             'Cleaner residual diff vs zenith', 'sum residual (STD - CNN) counts')
grouped_hist(axs[1, 1], cleaner_metrics['sum_residual_diff'], cleaner_metrics['group_theta'],
             'Cleaner residual diff vs theta bins', 'sum residual (STD - CNN) counts')

plt.tight_layout()
plt.show()


In [ ]:
# CLEANER: residual degradation scatter views

def scatter_with_categories(ax, x, y, xlabel, ylabel, title, categorical=False):
    y = pd.Series(y).astype(float)
    x = pd.Series(x)
    mask = np.isfinite(y) & x.notna()
    x = x[mask]
    y = y[mask]

    if not categorical:
        xv = pd.to_numeric(x, errors='coerce')
        m2 = np.isfinite(xv)
        xv = xv[m2]
        yv = y[m2]
        ax.scatter(xv, yv, s=10, alpha=0.45)
        ax.set_xlabel(xlabel, fontsize=FS)
        ax.set_ylabel(ylabel, fontsize=FS)
        ax.set_title(title, fontsize=FS)
        ax.grid(alpha=0.3)
        return

    cats = pd.Series(x.astype(str))
    uniq = [u for u in pd.unique(cats) if u != 'nan']
    mapper = {c: i for i, c in enumerate(uniq)}
    xi = cats.map(mapper).astype(float)
    jitter = np.random.default_rng(42).normal(0, 0.06, size=len(xi))
    ax.scatter(xi + jitter, y, s=12, alpha=0.45)
    ax.set_xticks(range(len(uniq)))
    ax.set_xticklabels(uniq, rotation=25)
    ax.set_xlabel(xlabel, fontsize=FS)
    ax.set_ylabel(ylabel, fontsize=FS)
    ax.set_title(title, fontsize=FS)
    ax.grid(alpha=0.3)

fig, axs = plt.subplots(2, 2, figsize=(16, 12))

scatter_with_categories(
    axs[0, 0], cleaner_metrics['snr_tag'], cleaner_metrics['sum_residual_diff'],
    'SNR', 'sum residual (STD - CNN) counts', 'Cleaner residual diff vs SNR', categorical=False
)
scatter_with_categories(
    axs[0, 1], cleaner_metrics['nbs_tag'], cleaner_metrics['sum_residual_diff'],
    'NBS tag', 'sum residual (STD - CNN) counts', 'Cleaner residual diff vs NBS', categorical=True
)
scatter_with_categories(
    axs[1, 0], cleaner_metrics['zenith_tag'], cleaner_metrics['sum_residual_diff'],
    'zenith tag', 'sum residual (STD - CNN) counts', 'Cleaner residual diff vs zenith', categorical=True
)
scatter_with_categories(
    axs[1, 1], cleaner_metrics['theta_tag'], cleaner_metrics['sum_residual_diff'],
    'theta / alpha / offset', 'sum residual (STD - CNN) counts', 'Cleaner residual diff vs theta', categorical=False
)

plt.tight_layout()
plt.show()

In [ ]:
# CLEANER: ON-region (source excess) degradation views
fig, axs = plt.subplots(2, 2, figsize=(16, 12))

grouped_hist(axs[0, 0], cleaner_metrics['sum_on_diff'], cleaner_metrics['group_snr'],
             'Cleaner ON diff vs SNR bins', 'sum ON (STD - CNN) counts')
grouped_hist(axs[0, 1], cleaner_metrics['sum_on_diff'], cleaner_metrics['group_nbs'],
             'Cleaner ON diff vs NBS tag', 'sum ON (STD - CNN) counts')
grouped_hist(axs[1, 0], cleaner_metrics['sum_on_diff'], cleaner_metrics['group_zenith'],
             'Cleaner ON diff vs zenith', 'sum ON (STD - CNN) counts')
grouped_hist(axs[1, 1], cleaner_metrics['sum_on_diff'], cleaner_metrics['group_theta'],
             'Cleaner ON diff vs theta bins', 'sum ON (STD - CNN) counts')

plt.tight_layout()
plt.show()


In [ ]:
# CLEANER: ON-region (source excess) scatter views

fig, axs = plt.subplots(2, 2, figsize=(16, 12))

scatter_with_categories(
    axs[0, 0], cleaner_metrics['snr_tag'], cleaner_metrics['sum_on_diff'],
    'SNR', 'sum ON (STD - CNN) counts', 'Cleaner ON diff vs SNR', categorical=False
)
scatter_with_categories(
    axs[0, 1], cleaner_metrics['nbs_tag'], cleaner_metrics['sum_on_diff'],
    'NBS tag', 'sum ON (STD - CNN) counts', 'Cleaner ON diff vs NBS', categorical=True
)
scatter_with_categories(
    axs[1, 0], cleaner_metrics['zenith_tag'], cleaner_metrics['sum_on_diff'],
    'zenith tag', 'sum ON (STD - CNN) counts', 'Cleaner ON diff vs zenith', categorical=True
)
scatter_with_categories(
    axs[1, 1], cleaner_metrics['theta_tag'], cleaner_metrics['sum_on_diff'],
    'theta / alpha / offset', 'sum ON (STD - CNN) counts', 'Cleaner ON diff vs theta', categorical=False
)

plt.tight_layout()
plt.show()

In [ ]:
# REGRESSOR: localization error degradation views
fig, axs = plt.subplots(2, 2, figsize=(16, 12))

grouped_hist(axs[0, 0], reg_metrics['err_deg'], reg_metrics['group_snr'],
             'Regressor error vs SNR bins', 'angular separation (deg)')
grouped_hist(axs[0, 1], reg_metrics['err_deg'], reg_metrics['group_nbs'],
             'Regressor error vs NBS tag', 'angular separation (deg)')
grouped_hist(axs[1, 0], reg_metrics['err_deg'], reg_metrics['group_zenith'],
             'Regressor error vs zenith', 'angular separation (deg)')
grouped_hist(axs[1, 1], reg_metrics['err_deg'], reg_metrics['group_theta'],
             'Regressor error vs theta bins', 'angular separation (deg)')

plt.tight_layout()
plt.show()


In [ ]:
# REGRESSOR: localization error scatter views

fig, axs = plt.subplots(2, 2, figsize=(16, 12))

scatter_with_categories(
    axs[0, 0], reg_metrics['snr_tag'], reg_metrics['err_deg'],
    'SNR', 'angular separation (deg)', 'Regressor error vs SNR', categorical=False
)
scatter_with_categories(
    axs[0, 1], reg_metrics['nbs_tag'], reg_metrics['err_deg'],
    'NBS tag', 'angular separation (deg)', 'Regressor error vs NBS', categorical=True
)
scatter_with_categories(
    axs[1, 0], reg_metrics['zenith_tag'], reg_metrics['err_deg'],
    'zenith tag', 'angular separation (deg)', 'Regressor error vs zenith', categorical=True
)
scatter_with_categories(
    axs[1, 1], reg_metrics['theta_tag'], reg_metrics['err_deg'],
    'theta / alpha / offset', 'angular separation (deg)', 'Regressor error vs theta', categorical=False
)

plt.tight_layout()
plt.show()

In [ ]:
# Optional: quick aggregate diagnostics by group

def quick_group_stats(df, metric, group):
    out = (
        df[[metric, group]]
        .dropna()
        .groupby(group)[metric]
        .agg(['count', 'mean', 'median', 'std'])
        .sort_values('median')
    )
    return out

print('Cleaner residual diff by zenith')
display(quick_group_stats(cleaner_metrics, 'sum_residual_diff', 'group_zenith'))

print('Cleaner ON diff by zenith')
display(quick_group_stats(cleaner_metrics, 'sum_on_diff', 'group_zenith'))

print('Regressor err by zenith')
display(quick_group_stats(reg_metrics, 'err_deg', 'group_zenith'))
